<a href="https://colab.research.google.com/github/Boran-Sert/SoftITO_Projler/blob/master/Python/20_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [52]:
torch.manual_seed(42)
torch.set_printoptions(precision=2, sci_mode=True)

In [53]:
def olcekli_dikkat(Q,K,V,maske= None):
  # Tüm anahtar vektör değerlini alabilmek için son endeksden başlıyoruz
  d_k = Q.size(-1)

  # Her sorgu ile her anahtarın benzerliği ölçmek için √d_k ile gösteririz
  skorlar = Q @ K.transpose(-2,-1)/math.sqrt(d_k)

  # Maskeli değerler var mı yok mu onu kontrol etmek eğer maskeli alanlar varsa hesabı karıştırmasın diye
  # - sonsuz burakıo sonrasında -sonsuz algoritmadı softmax ile olur

  if maske is not None:
    skorlar.masked_fill(maske==0, float("-inf"))

  # Skorları ağırlıklara çevirmek için
  agirliklar = F.softmax(skorlar, dim = -1)
  cikti = agirliklar@V
  return cikti, agirliklar


In [54]:
# ============================== MASKE OLMAZSA =================================
def olcekli_dikkat(Q,K,V,maske=None):
  # Tüm anahtar vektör değerlini alabilmek için son endeksden başlıyoruz
  d_k = Q.size(-1)

  # Her sorgu ile her anahtarın benzerliği ölçmek için √d_k ile gösteririz
  skorlar = Q @ K.transpose(-2,-1)/math.sqrt(d_k)

  if maske is not None:
    skorlar.masked_fill(maske==0, float("-inf"))

  # Skorları ağırlıklara çevirmek için
  agirliklar = F.softmax(skorlar, dim = -1)
  cikti = agirliklar@V
  return cikti, agirliklar

In [55]:
# her satır bir kelime vektörü
x = torch.tensor([[1.0,0.0,1.0,0.0],[0.0,2.0,0.0,2.0],[1.0,1.0,1.0,1.0]])
cikti,agirliklar = olcekli_dikkat(x,x,x)
print(cikti)
print(agirliklar)

tensor([[8.45e-01, 7.33e-01, 8.45e-01, 7.33e-01],
        [1.33e-01, 1.85e+00, 1.33e-01, 1.85e+00],
        [5.78e-01, 1.27e+00, 5.78e-01, 1.27e+00]])
tensor([[4.22e-01, 1.55e-01, 4.22e-01],
        [1.59e-02, 8.67e-01, 1.17e-01],
        [1.55e-01, 4.22e-01, 4.22e-01]])


In [56]:
class CokKafaliDikkat(nn.Module):
  def __init__(self,d_model,n_kafa):
    super().__init__()
    assert d_model % n_kafa == 0
    self.n_kafa = n_kafa
    self.d_model = d_model
    self.d_k = d_model // n_kafa

    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)

    self.son_agirliklar = None

  def kafalara_bol (self,t):
    B, L,_ = t.shape
    return t.view(B, L, self.n_kafa, self.d_k).transpose(1,2)

  def forward(self,x,maske=None):
    B, L, d_model = x.shape
    Q = self.kafalara_bol(self.W_q(x))
    K = self.kafalara_bol(self.W_k(x))
    V = self.kafalara_bol(self.W_v(x))

    cikti,agirliklar = olcekli_dikkat(Q,K,V,maske)
    self.son_agirliklar = agirliklar.detach()

    cikti = cikti.transpose(1,2).reshape(B, L, d_model)
    return self.W_o(cikti)

In [57]:
deneme = torch.rand(2,5,64)
mha = CokKafaliDikkat(64,8)
cikti = mha(deneme)
print(mha.son_agirliklar.shape)
print(cikti.shape)
print(mha.son_agirliklar)

torch.Size([2, 8, 5, 5])
torch.Size([2, 5, 64])
tensor([[[[2.12e-01, 2.13e-01, 1.89e-01, 1.90e-01, 1.97e-01],
          [2.02e-01, 2.00e-01, 1.94e-01, 2.00e-01, 2.04e-01],
          [2.15e-01, 2.06e-01, 1.89e-01, 1.92e-01, 1.98e-01],
          [2.09e-01, 2.08e-01, 1.94e-01, 1.92e-01, 1.96e-01],
          [2.09e-01, 2.03e-01, 1.97e-01, 1.92e-01, 2.00e-01]],

         [[2.04e-01, 1.97e-01, 2.00e-01, 2.13e-01, 1.88e-01],
          [2.02e-01, 1.96e-01, 2.01e-01, 2.07e-01, 1.94e-01],
          [2.00e-01, 1.98e-01, 2.02e-01, 2.06e-01, 1.94e-01],
          [1.94e-01, 2.01e-01, 2.05e-01, 2.02e-01, 1.97e-01],
          [1.95e-01, 2.06e-01, 2.05e-01, 1.95e-01, 1.99e-01]],

         [[2.18e-01, 1.90e-01, 1.95e-01, 2.04e-01, 1.93e-01],
          [2.10e-01, 2.04e-01, 1.94e-01, 1.94e-01, 1.98e-01],
          [2.24e-01, 1.99e-01, 1.91e-01, 2.02e-01, 1.83e-01],
          [2.20e-01, 1.94e-01, 1.95e-01, 2.01e-01, 1.89e-01],
          [2.03e-01, 2.05e-01, 1.99e-01, 1.99e-01, 1.94e-01]],

         [[1.89e

In [58]:
class KonumKodlaması(nn.Module):
  def __init__(self,d_model,max_len=1000):
    super().__init__()
    pe = torch.zeros(max_len,d_model)
    konum = torch.arange(max_len).unsqueeze(1).float()
    freakns = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000)/d_model))
    pe[:,0::2] = torch.sin(konum*freakns)
    pe[:,1::2] = torch.cos(konum*freakns)
    self.register_buffer("pe",pe.unsqueeze(0))

  def forward(self,x):
    return x+self.pe[:,:x.size(1)]

In [66]:
class IleriBesleme(nn.Module):
  def __init__(self,d_model,d_ff):
    super().__init__()
    self.ff = nn.Sequential(
        #boyut genişletmek için
        nn.Linear(d_model,d_ff),
        # - değerleri sıfıra çekmek için
        nn.ReLU(),
        # Model boyutuna çevirme
        nn.Linear(d_ff,d_model)
    )
  def forward(self,x):
    return self.ff(x)

In [60]:
class KodlayıcıBlok(nn.Module):
  def __init__(self,d_model,n_kafa,d_ff,dropout=0.1):
    super().__init__()
    self.attention = CokKafaliDikkat(d_model,n_kafa)
    self.ffn = IleriBesleme(d_model,d_ff)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x,maske=None):
    x = self.norm1(x+self.dropout(self.attention(x,maske)))
    x = self.norm2(x+self.dropout(self.ffn(x)))

    return x


In [65]:
class MiniTransformer(nn.Module):
  def __init__(self,sozluk_boyutu,d_model =64, n_kafa=4, d_ff=128, max_len=50, n_katman=2):
    super().__init__()
    self.gomme= nn.Embedding(sozluk_boyutu,d_model)
    self.konum_kodlayici = KonumKodlaması(d_model,max_len)
    self.bloklar = nn.ModuleList([KodlayıcıBlok(d_model,n_kafa,d_ff) for _ in range(n_katman)])
    self.cikis = nn.Linear(d_model,sozluk_boyutu)

  def forward(self,x,maske=None):
    x = self.gomme(x)
    x = self.konum_kodlayici(x)
    for blok in self.bloklar:
      x = blok(x,maske)
    return self.cikis(x)

In [67]:
SOZLUK = 10
model = MiniTransformer(sozluk_boyutu=SOZLUK)
toplam = sum(p.numel() for p in model.parameters())
print(f"Modelimiz {toplam:,} parametreye sahiptir")

Modelimiz 68,234 parametreye sahiptir


In [63]:
uzunluk = 8
def veri_uret(adet):
  x= torch.randint(0,SOZLUK,(adet,uzunluk))
  y = torch.sort(x,dim=1).values
  return x,y

ornek_x, ornek_y = veri_uret(1)
print("Girdi:" ,ornek_x[0].tolist())
print("Hedef:", ornek_y[0].tolist())

Girdi: [8, 4, 0, 9, 8, 2, 6, 3]
Hedef: [0, 2, 3, 4, 6, 8, 8, 9]


In [68]:
optimzer = torch.optim.Adam(model.parameters(),lr=1e-3)
kayip_fn = nn.CrossEntropyLoss()
kayiplar = []

model.train()

for adim in range(1,2001):
  x,y= veri_uret(64)
  logits= model(x)
  kayip = kayip_fn(logits.reshape(-1,SOZLUK),y.reshape(-1))
  optimzer.zero_grad()
  kayip.backward()
  optimzer.step()
  kayiplar.append(kayip.item())

  if adim %200 == 0:
    print(f"Adım {adim:4d} | Kayıp: {kayip.item():.4f}")

Adım  200 | Kayıp: 0.0996
Adım  400 | Kayıp: 0.0329
Adım  600 | Kayıp: 0.0171
Adım  800 | Kayıp: 0.0068
Adım 1000 | Kayıp: 0.0143
Adım 1200 | Kayıp: 0.0039
Adım 1400 | Kayıp: 0.0061
Adım 1600 | Kayıp: 0.0013
Adım 1800 | Kayıp: 0.0046
Adım 2000 | Kayıp: 0.0016


In [70]:
model.eval()
with torch.no_grad():
  x_test, y_test = veri_uret(100000)
  tahmin = model(x_test).argmax(dim=-1)

rakam_dogruluk = (tahmin ==y_test).float().mean().item()
dizi_dogruluk = (tahmin == y_test).all(dim=1).float().mean().item()

print(f"Rakam Doğruluk: {rakam_dogruluk*100:.2f}%")
print(f"Dizi Doğruluk: {dizi_dogruluk*100:.2f}%")

for i in range(2):
  print("Girdi:",x_test[i].tolist())
  print("Hedef:",y_test[i].tolist())
  print("Tahmin:",tahmin[i].tolist())
  print()

Rakam Doğruluk: 99.96%
Dizi Doğruluk: 99.69%
Girdi: [3, 2, 2, 6, 1, 0, 4, 9]
Hedef: [0, 1, 2, 2, 3, 4, 6, 9]
Tahmin: [0, 1, 2, 2, 3, 4, 6, 9]

Girdi: [0, 2, 3, 8, 7, 7, 8, 0]
Hedef: [0, 0, 2, 3, 7, 7, 8, 8]
Tahmin: [0, 0, 2, 3, 7, 7, 8, 8]

